# DX702 Coding Quiz: Week 4

In [12]:
import pandas as pd
import numpy as np
import statsmodels.api as sm # For linear regression
import statsmodels.formula.api as smf
# import scipy.stats as stats
# import seaborn as sns

# from scipy.stats import skew
# from sklearn.linear_model import LinearRegression

## Task: Given data about an instrumental variable, find the effect.

X is the treatment, W the confounder,

Y the outcome, and Z the instrument.

Use homework_4.1.csv. 

In [13]:
df_4_1 = pd.read_csv('homework_4.1.csv')
df_4_1.drop(columns=['Unnamed: 0'], inplace = True)
print(df_4_1.info())
df_4_1.head(n = 10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Z       5000 non-null   int64  
 1   W       5000 non-null   float64
 2   X       5000 non-null   float64
 3   Y       5000 non-null   float64
dtypes: float64(3), int64(1)
memory usage: 156.4 KB
None


,Z,W,X,Y
0,0,-0.155644,-0.496971,0.282484
1,1,0.529539,2.284240,4.740596
2,1,0.910514,0.872232,3.449569
3,1,-0.705476,2.157260,3.002531
4,0,-0.590874,-0.386730,-1.848796
5,1,0.533635,0.981649,1.866566
6,0,1.293241,-0.041748,2.427692
7,0,0.243445,-0.113452,-0.177196
8,0,-0.714465,-0.129564,1.673641
9,1,-1.548130,0.126844,-0.379818


### Question 1: As in the explanation, try averaging the ﻿﻿Y difference and ﻿﻿ X difference (over W﻿﻿ and Z﻿﻿) in two ways: 
1.) Subtract the average ﻿﻿ Y value for Z = 1 ﻿﻿ and Z = 0﻿﻿. Subtract the average ﻿﻿ X value for ﻿﻿Z = 1 and Z = 0 ﻿﻿. Divide the two. 

2.) Find the average ﻿﻿ Y value for Z = 1 ﻿﻿ and Z = 0﻿﻿ for a narrow range of ﻿﻿W. Find the average ﻿﻿X value for ﻿﻿Z = 1 and ﻿﻿ Z = 0 for the same narrow range of ﻿﻿W . Take the ratio to find the effect. Then average this over all the ranges of ﻿﻿W. 

For the first item, the effect is closest to: 



Option A: 1

Option B: 0.5 

<font color='magenta'>**Option C : 1.5**</font>

Option D: 2

In [14]:
# METHOD 1 ------------------------------------------------------------

# average Y for Z=1 and Z=0
avg_y_z1 = df_4_1[df_4_1['Z'] == 1]['Y'].mean()
avg_y_z0 = df_4_1[df_4_1['Z'] == 0]['Y'].mean()

# average X for Z=1 and Z=0
avg_x_z1 = df_4_1[df_4_1['Z'] == 1]['X'].mean()
avg_x_z0 = df_4_1[df_4_1['Z'] == 0]['X'].mean()

# differences
diff_y = avg_y_z1 - avg_y_z0
diff_x = avg_x_z1 - avg_x_z0

# effect 
effect_method_1 = diff_y / diff_x

print(f"Average Y for Z=1: {avg_y_z1}")
print(f"Average Y for Z=0: {avg_y_z0}")
print(f"Average X for Z=1: {avg_x_z1}")
print(f"Average X for Z=0: {avg_x_z0}")
print(f"Difference in Y: {diff_y}")
print(f"Difference in X: {diff_x}")
print(f"Effect using Method 1: {effect_method_1}")

Average Y for Z=1: 1.5334744841635137
Average Y for Z=0: -0.05737074202839066
Average X for Z=1: 1.0091964255735302
Average X for Z=0: -0.009362563571315565
Difference in Y: 1.5908452261919044
Difference in X: 1.0185589891448457
Effect using Method 1: 1.5618587073955674


In [15]:

# Determine the range of W
min_w = df_4_1['W'].min()
max_w = df_4_1['W'].max()

print(f"Min W: {min_w}")
print(f"Max W: {max_w}")

# create 10 bins for W
num_bins        = 10
bins            = np.linspace(min_w, max_w, num_bins + 1)
df_4_1['W_bin'] = pd.cut(df_4_1['W'], bins, include_lowest = True, labels = False)

effect_ratios   = []

# Iterate through each W bin
for bin_label in sorted(df_4_1['W_bin'].unique()):
    df_bin = df_4_1[df_4_1['W_bin'] == bin_label]

    # Ensure there are both Z=0 and Z=1 in the bin
    if len(df_bin[df_bin['Z'] == 1]) > 0 and len(df_bin[df_bin['Z'] == 0]) > 0:
        avg_y_z1_bin = df_bin[df_bin['Z'] == 1]['Y'].mean()
        avg_y_z0_bin = df_bin[df_bin['Z'] == 0]['Y'].mean()
        avg_x_z1_bin = df_bin[df_bin['Z'] == 1]['X'].mean()
        avg_x_z0_bin = df_bin[df_bin['Z'] == 0]['X'].mean()

        diff_y_bin = avg_y_z1_bin - avg_y_z0_bin
        diff_x_bin = avg_x_z1_bin - avg_x_z0_bin

        # Avoid division by zero
        if diff_x_bin != 0:
            effect_ratios.append(diff_y_bin / diff_x_bin)

# Average of the ratios
effect_method_2 = np.mean(effect_ratios)

print(f"Effect using Method 2 (averaged over {num_bins} W bins): {effect_method_2:.6f}")

Min W: -3.3034061894656137
Max W: 4.783256692238601
Effect using Method 2 (averaged over 10 W bins): 0.908542


### Question 2: do you need to know W to do this? Yes or No .

<font color='plum'>NO. Assuming W is the confounder, then Method 1 doesn't require it to calculate the effect.

## Task: 

Given student data involving test scores (﻿﻿X), a cut-off, and an outcome (Y), which measures whether the students got into college (as in the example in the text), determine whether the math course helps students get into college in each dataset. Use datasets homework_4.2.a and homework_4.2.b. 

In [16]:
df_4_2a = pd.read_csv("https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/main/homework_4.2.a.csv")
df_4_2a.drop(columns=['Unnamed: 0'], inplace = True)

print(df_4_2a.info())
df_4_2a.head(n = 10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   X       100000 non-null  float64
 1   Y       100000 non-null  int64  
dtypes: float64(1), int64(1)
memory usage: 1.5 MB
None


,X,Y
0,81.822339,1
1,92.487870,0
2,85.372460,0
3,78.828025,0
4,75.807080,1
5,92.393683,0
6,80.254335,1
7,86.296267,0
8,76.296099,0
9,89.144863,1


In [17]:
df_4_2b = pd.read_csv("https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/main/homework_4.2.b.csv")
df_4_2b.drop(columns=['Unnamed: 0'], inplace = True)

print(df_4_2b.info())
df_4_2b.head(n = 10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   X2      100000 non-null  float64
 1   Y2      100000 non-null  int64  
dtypes: float64(1), int64(1)
memory usage: 1.5 MB
None


,X2,Y2
0,76.643034,1
1,87.743397,1
2,81.639469,1
3,73.740485,0
4,90.480268,1
5,82.092415,1
6,70.021280,0
7,89.993287,1
8,78.743872,1
9,69.328605,0


In [18]:
# Define the cut-off score
cutoff = 80

In [19]:
# --- Analysis for Dataset 'a' (X, Y) ---
X_col_a = 'X'
Y_col_a = 'Y'

# Create dummy variable and distance from cut-off for dataset 'a'
df_4_2a['above_cutoff']        = (df_4_2a[X_col_a] >= cutoff).astype(int)
df_4_2a['dist_from_cutoff']    = df_4_2a[X_col_a] - cutoff

# Split data for before and after cut-off for dataset 'a'
df_a_before_cutoff  = df_4_2a[df_4_2a[X_col_a] < cutoff]
df_a_after_cutoff   = df_4_2a[df_4_2a[X_col_a] >= cutoff]

# Calculate slopes for dataset 'a'
model_a_before = smf.ols(f'{Y_col_a} ~ {X_col_a}', data = df_a_before_cutoff).fit()
slope_a_before = model_a_before.params[X_col_a]
print(f"Dataset 'a' - Slope of Y before cut-off: {slope_a_before}")

model_a_after = smf.ols(f'{Y_col_a} ~ {X_col_a}', data = df_a_after_cutoff).fit()
slope_a_after = model_a_after.params[X_col_a]
print(f"Dataset 'a' - Slope of Y after cut-off: {slope_a_after}")


Dataset 'a' - Slope of Y before cut-off: 0.00022383079703675835
Dataset 'a' - Slope of Y after cut-off: 0.00016108686482244584


In [20]:

# --- Analysis for Dataset 'b' (X2, Y2) ---
X_col_b = 'X2'
Y_col_b = 'Y2'

# Create dummy variable and distance from cut-off for dataset 'b'
df_4_2b['above_cutoff'] = (df_4_2b[X_col_b] >= cutoff).astype(int)
df_4_2b['dist_from_cutoff'] = df_4_2b[X_col_b] - cutoff

# Split data for before and after cut-off for dataset 'b'
df_b_before_cutoff  = df_4_2b[df_4_2b[X_col_b] < cutoff]
df_b_after_cutoff   = df_4_2b[df_4_2b[X_col_b] >= cutoff]

# Calculate slopes for dataset 'b'
model_b_before = smf.ols(f'{Y_col_b} ~ {X_col_b}', 
                         data = df_b_before_cutoff).fit()

slope_b_before = model_b_before.params[X_col_b]
print(f"Dataset 'b' - Slope of Y before cut-off: {slope_b_before}")

model_b_after = smf.ols(f'{Y_col_b} ~ {X_col_b}', 
                        data = df_b_after_cutoff).fit()

slope_b_after = model_b_after.params[X_col_b]
print(f"Dataset 'b' - Slope of Y after cut-off: {slope_b_after}")


Dataset 'b' - Slope of Y before cut-off: 0.010216711274456683
Dataset 'b' - Slope of Y after cut-off: 0.0050086140682368005


In [21]:

# --- RDD Models for Question 5 ---
print("\n--- RDD Model for Dataset 'a' ---")

rdd_model_a = smf.ols(f'{Y_col_a} ~ dist_from_cutoff + above_cutoff + dist_from_cutoff:above_cutoff', 
                      data = df_4_2a).fit()
print(rdd_model_a.summary())

print("\n--- RDD Model for Dataset 'b' ---")
rdd_model_b = smf.ols(f'{Y_col_b} ~ dist_from_cutoff + above_cutoff + dist_from_cutoff:above_cutoff',
                       data= df_4_2b).fit()

print(rdd_model_b.summary())


--- RDD Model for Dataset 'a' ---
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.090
Model:                            OLS   Adj. R-squared:                  0.090
Method:                 Least Squares   F-statistic:                     3300.
Date:                Mon, 09 Jun 2025   Prob (F-statistic):               0.00
Time:                        11:38:51   Log-Likelihood:                -67412.
No. Observations:              100000   AIC:                         1.348e+05
Df Residuals:                   99996   BIC:                         1.349e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------

### Question 3: In this dataset, is Y increasing or decreasing before the cut-off ?


<font color='plum'>**INCREASING**. For both Datasets A and B, the slope of Y with respect to X (or X2) before the cut-off of 80 was positive, indicating that Y is increasing as X increases before the cut-off.

### Question 4: is Y's slope higher or lower after the cut-off compared with before ?

<font color = 'plum'>**LOWER**. For both Datasets 'a' and 'b', the calculated slope of Y with respect to X (or X2) after the cut-off was lower than the slope before the cut-off.

### Question 5: Given a cut-off score of 80, which dataset seems most likely to involve a nonzero linear term, allowing Y﻿﻿ to relate linearly to X﻿﻿ before and after the cut-off? 
Option A: Dataset b (﻿﻿X2, Y2﻿﻿)

Option B: Dataset a ( X﻿﻿, Y ﻿﻿)

<font color='plum'>**ANSWER = Option A: Dataset b (X2, Y2)**

Based on the Regression Discontinuity Design (RDD) model summaries:

For Dataset 'a' (X, Y), the `dist_from_cut-off` and `dist_from_cut-off:above_cut-off` terms (representing the linear relationship before and the change in slope after the cut-off) had high p-values (0.525 and 0.904, respectively). This suggests that these linear terms are not statistically significant, implying a very weak or negligible linear relationship.


For Dataset 'b' (X2, Y2), both the `dist_from_cut-off` and `dist_from_cut-off:above_cut-off` terms were highly statistically significant (p-values of 0.000). This indicates a clear and statistically significant linear relationship between Y2 and X2, both before and after the cut-off, making it more likely to involve a nonzero linear term.